In [170]:
import pandas as pd
import numpy as np

In [171]:
acc1 = pd.read_csv('../data/acc1.csv')
acc2 = pd.read_csv('../data/acc2.csv')

In [172]:
acc1["Account"] = "Chequing"
acc2["Account"] = "Savings"

bank_df_raw = pd.concat([acc1, acc2], ignore_index=True)

### First filter (coolumns)

In [173]:
bank_df = bank_df_raw.drop(columns=["Filter", "Type of Transaction"])
bank_df.columns

Index(['Date', 'Description', 'Sub-description', 'Amount', 'Balance',
       'Account'],
      dtype='object')

### Second filter (removing internal trasactions)

In [174]:
bank_df = bank_df[bank_df["Description"] != "customer transfer cr."]
bank_df = bank_df[bank_df["Description"] != "customer transfer dr."]

In [175]:
bank_df["Sub-description"] = (
    bank_df["Sub-description"]
    .fillna("none")         # fill NaN
    .replace("", "none")    # fill empty strings
)

# fill when there is just spaces 
bank_df.loc[bank_df["Sub-description"].str.strip() == "", "Sub-description"] = "none"

bank_df[bank_df["Description"] == "deposit"]

,Date,Description,Sub-description,Amount,Balance,Account
300,2025-05-26,deposit,Free Interac E-Transfer,60.00,4939.80,Chequing
326,2025-05-17,deposit,Free Interac E-Transfer,8.00,5146.12,Chequing
349,2025-05-12,deposit,Free Interac E-Transfer,5899.00,5899.24,Chequing
359,2025-05-08,deposit,none,1000.00,1000.00,Chequing
360,2025-05-08,deposit,none,0.00,0.00,Chequing
362,2025-09-19,deposit,Free Interac E-Transfer,24.00,2439.38,Savings
363,2025-09-19,deposit,Free Interac E-Transfer,11.53,2415.38,Savings
365,2025-09-17,deposit,Free Interac E-Transfer,434.72,2838.85,Savings
367,2025-09-04,deposit,Free Interac E-Transfer,423.19,2828.13,Savings
370,2025-08-29,deposit,Free Interac E-Transfer,20.00,3829.91,Savings


### Dropping the initial deposits and parent's deposits

In [176]:
bank_df = bank_df[~((bank_df["Description"] == "deposit") & (bank_df["Sub-description"] == "none"))]
bank_df = bank_df[bank_df["Description"] != "abm deposit"]
bank_df = bank_df.drop(index=[400, 397, 389, 349])
bank_df[bank_df["Description"] == "deposit"]

,Date,Description,Sub-description,Amount,Balance,Account
300,2025-05-26,deposit,Free Interac E-Transfer,60.00,4939.80,Chequing
326,2025-05-17,deposit,Free Interac E-Transfer,8.00,5146.12,Chequing
362,2025-09-19,deposit,Free Interac E-Transfer,24.00,2439.38,Savings
363,2025-09-19,deposit,Free Interac E-Transfer,11.53,2415.38,Savings
365,2025-09-17,deposit,Free Interac E-Transfer,434.72,2838.85,Savings
367,2025-09-04,deposit,Free Interac E-Transfer,423.19,2828.13,Savings
370,2025-08-29,deposit,Free Interac E-Transfer,20.00,3829.91,Savings
372,2025-08-21,deposit,Free Interac E-Transfer,428.96,4314.91,Savings
374,2025-08-07,deposit,Free Interac E-Transfer,423.19,4310.95,Savings
379,2025-07-24,deposit,Free Interac E-Transfer,423.19,5697.72,Savings


In [177]:
# --- Normalização ---
desc = bank_df["Description"].astype(str).str.strip().str.lower()
subd = (
    bank_df["Sub-description"]
    .fillna("")
    .astype(str)
    .str.replace("\u00A0", " ", regex=False)
    .str.strip()
)
subd_lower = subd.str.lower()

# --- Classes (Earnings vs Expenses) ---
earnings_set = {"payroll deposit", "correction", "interest", "deposit"}
bank_df["Class"] = np.where(desc.isin(earnings_set), "Earnings", "Expenses")

# --- Category (macro) ---
bank_df["Category"] = "Others"  # default
bank_df["Sub-Category"] = "None"  # default

# payroll deposit
bank_df.loc[desc.eq("payroll deposit"), "Category"] = "Payment"

# withdrawal
mask_withdraw = desc.eq("withdrawal")
# 1) índices específicos → shopping
bank_df.loc[bank_df.index.isin([399, 352, 289]) & mask_withdraw, "Category"] = "shopping"
# 2) caso específico → Bills - Rent
bank_df.loc[mask_withdraw & (bank_df["Amount"] == 1200), ["Category","Sub-Category"]] = ["Bills","Rent"]
# 3) todo o resto → money sent
bank_df.loc[mask_withdraw & (bank_df["Category"] == "Others"), "Category"] = "money sent"

# pos purchase
mask_pos = desc.eq("pos purchase")
bank_df.loc[mask_pos & subd.str.startswith("Compass", na=False), ["Category","Sub-Category"]] = ["Bills","Transport"]
bank_df.loc[mask_pos & subd.str.startswith("Walmart", na=False), "Category"] = "Groceries"
bank_df.loc[mask_pos & subd.str.startswith("Real Cdn", na=False), "Category"] = "Groceries"

# bill payment
bank_df.loc[desc.eq("bill payment"), ["Category","Sub-Category"]] = ["Bills","Cellphone"]

# correction & interest
bank_df.loc[desc.isin(["correction", "interest"]), "Category"] = "earnings"

# deposit
mask_dep = desc.eq("deposit")
bank_df.loc[mask_dep & (bank_df["Amount"].between(300, 900)), "Category"] = "Payment"
bank_df.loc[mask_dep & ~bank_df["Amount"].between(300, 900), "Category"] = "shared bills"

# service charge
bank_df.loc[desc.eq("service charge"), ["Category","Sub-Category"]] = ["Bills","Bank"]


In [178]:
bank_df['Description'].unique()

array(['payroll deposit', 'withdrawal', 'pos purchase', 'bill payment',
       'correction', 'deposit', 'interest', 'service charge'],
      dtype=object)

In [179]:
bank_df[bank_df["Description"] == "withdrawal"]

,Date,Description,Sub-description,Amount,Balance,Account,Class,Category,Sub-Category
1,2025-09-25,withdrawal,Free Interac E-Transfer,-15.3,4272.57,Chequing,Expenses,money sent,None
3,2025-09-23,withdrawal,Free Interac E-Transfer,-13.0,4293.81,Chequing,Expenses,money sent,None
17,2025-09-19,withdrawal,Free Interac E-Transfer,-18.0,4417.14,Chequing,Expenses,money sent,None
72,2025-09-02,withdrawal,Free Interac E-Transfer,-1200.0,3959.04,Chequing,Expenses,money sent,None
131,2025-08-06,withdrawal,Free Interac E-Transfer,-45.0,2789.05,Chequing,Expenses,money sent,None
136,2025-08-04,withdrawal,Free Interac E-Transfer,-7.0,2891.67,Chequing,Expenses,money sent,None
143,2025-08-01,withdrawal,Free Interac E-Transfer,-31.0,2963.71,Chequing,Expenses,money sent,None
146,2025-07-31,withdrawal,Free Interac E-Transfer,-1200.0,2545.26,Chequing,Expenses,money sent,None
207,2025-06-30,withdrawal,Free Interac E-Transfer,-1200.0,1113.21,Chequing,Expenses,money sent,None
243,2025-06-14,withdrawal,Free Interac E-Transfer,-20.0,917.66,Chequing,Expenses,money sent,None


### Problemas na classificação do aluguel. Lembrar de testar o tipo da coluna. Ultima solução do chat nao resolveu e é ela que esta aplicada na celula.

Temos que definir as regras pra classificação das compras feitas no POS como proxima etapa.